# Telephone Games Between AI Agents

A message is passed along a chain of AI agents. Each agent sees **only what the previous agent produced**:

1. Agent 1 reads a description of a person and fills in a 9-item personality survey (true / false).
2. Agent 2 sees only those answers and writes a new description in its own words.
3. Agent 3 reads that description and fills in the survey again, and so on.

At every survey step we compare the answers with the original person. Questions:
- How fast does the person get lost? Does it drop to **chance** (a coin flip gets 50% right)?
- Do people with **clashing traits** get "fixed" into a clean **stereotype**?
- Does that change with the **model**, the **model size**, **mixing models**, or the **temperature**?

**Setup:** put your Gemini API key in a file called `.env` in this folder (see `.env.example`).

**How this notebook is organised**
- *The person* and *Method details*: the questionnaire, the exact prompts, the measures and their chance levels.
- *Part 1* and *Your turn*: run chains live.
- *Part 2* and *Part 3*: the full experiment and its results (preloaded).
- *Export the data* and *Reuse this for your own study*: take the data to R or SPSS, and adapt the code to a new study.


**How to read the code**
- A notebook is a list of **cells**: text cells like this one, and **code cells** (the grey boxes). Click a code cell and press **Shift + Enter** to run it; its result appears right below.
- Run the cells **from top to bottom**: later cells use what earlier cells prepared.
- Names in **CAPITALS**, like `DEMO_TEMPERATURE = 1.0`, are **settings** you can change.
- `t.something` is a function from the file `telephone.py` (it runs the chains); `a.something` is a function from `analysis.py` (tables and charts). Both files are in this folder and can be opened in Jupyter.
- After each step, a **🔍 Look inside** cell prints the actual code of the functions used, with notes in plain words. Lines starting with `#` are those notes; the computer ignores them.

In [ ]:
import inspect
from pathlib import Path

import pandas as pd
from IPython.display import HTML
from pygments import highlight
from pygments.formatters import HtmlFormatter
from pygments.lexers import PythonLexer

import analysis as a
import telephone as t

try:
    caller = t.make_caller()
    print("Key found, ready.")
except RuntimeError as e:
    caller = None
    print(f"No API key: {e}\nThe preloaded results and charts still work; live chains need a key.")


def show_code(function_or_text):
    """Shows code with colours: a function's code, or a piece of code text (used in the 'Look inside' cells)."""
    style = (
        "font-family: Menlo, Consolas, 'DejaVu Sans Mono', monospace; font-size: 13px; line-height: 1.45; padding: 10px"
    )
    colours = HtmlFormatter(noclasses=True, style="friendly", prestyles=style)
    code = function_or_text if isinstance(function_or_text, str) else inspect.getsource(function_or_text)
    display(HTML(highlight(code, PythonLexer(), colours)))

## The person

9 statements from a short Big Five questionnaire (extraversion, conscientiousness, neuroticism).

- **Coherent profiles** (8): every high/low combination of the three traits, e.g. *anxious organized introvert*.
- **Random profiles**: fixed random answer patterns that clash (e.g. *reserved* but also *talkative*). They are at least 2 answers away from every coherent profile.

In [ ]:
PROFILES = t.PEOPLE  # the 10 people, written out in telephone.py

pd.DataFrame(
    {name: ["yes" if a else "no" for a in answers] for name, answers in PROFILES.items()},
    index=[pos for pos, _ in t.ITEMS],
)

### 🔍 Look inside: how the people are made

The 10 people are simply **written out** in `telephone.py`: one line per person, one answer per statement (1 = true, 0 = false), in the order of the questionnaire. `TRAIT_KEYS` says which statements belong to which trait and which are reverse-keyed. The 8 stereotypes are every high/low combination of the 3 traits; the 2 random people were picked once by coin flips (keeping only patterns at least 2 answers away from every stereotype) and are fixed. Running `python telephone.py` checks that every stereotype agrees with the keying.

In [ ]:
telephone_code = inspect.getsource(t)  # the text of the whole telephone.py file
show_code(telephone_code[telephone_code.index("# Which statements") : telephone_code.index("\nPEOPLE = ")])

## Method details

**The questionnaire.** 9 statements, 3 per trait. Answers are stored in item order as 1 = true, 0 = false. Statements marked (−) are reverse-keyed: *true* counts against the trait.

| Trait | Statements |
|---|---|
| Extraversion | is talkative · is reserved (−) · tends to be quiet (−) |
| Conscientiousness | does a thorough job · perseveres until the task is finished · tends to be disorganized (−) |
| Neuroticism | worries a lot · can be tense · is emotionally stable, not easily upset (−) |

The items and keying live in `telephone.py` (`ITEMS`, `TRAIT_KEYS`), next to the 10 people (`STEREOTYPES`, `RANDOM_PEOPLE`).

**The chain.** Agent 1 gets the original person as 9 plain sentences. Odd-numbered agents fill in the survey, even-numbered agents write a description. Each agent sees only the previous agent's output (no memory). If a survey reply cannot be read, the chain stops there and the failure is counted.

**The two prompts**, exactly as the agents receive them (shown here for *random-1*):

In [ ]:
example = PROFILES["random-1"]
print("=== SURVEY AGENT (odd steps) ===\n")
print(t.survey_prompt(t.profile_to_text(example)))
print("\n=== DESCRIPTION AGENT (even steps) ===\n")
print(t.description_prompt(example))

### 🔍 Look inside: how the prompts are built and the answers are read

`t.profile_to_text` writes a person as 9 sentences, `t.survey_prompt` and `t.description_prompt` wrap that in the instructions you just saw. When a survey reply comes back, `t.parse_answers` looks for the JSON part and accepts it only if it holds exactly 9 true/false values. Anything else counts as unreadable: nothing is guessed.

In [ ]:
show_code(t.profile_to_text)
show_code(t.survey_prompt)
show_code(t.description_prompt)
show_code(t.parse_answers)

**Measures**, scored after every survey step:
- **Accuracy**: share of the 9 answers that still match the original person.
- **Distance to the nearest stereotype**: how many answers you would need to change to reach one of the 8 coherent profiles (0 to 3). 0 = the person is now a clean stereotype.
- **Unreadable replies** are counted, never guessed. **Looping** flags descriptions that repeat the same sentence three or more times.
- **Text drift**: how similar each description is to the chain's first description, using embeddings (Part 3).

**Chance levels**: pure math, no data. What you would see if every agent ticked the 9 answers by coin flip: there are 2^9 = 512 possible answer patterns, and 8 of them are stereotypes. Compare every result against these, not against zero.

In [ ]:
print("If every answer were a coin flip (no chain needed, just counting):")
print(f"  Accuracy:                                   {t.CHANCE_ACCURACY:.2f}")
print(f"  Distance to the nearest stereotype:         {t.chance_stereotype_distance():.2f}  (scale 0 to 3)")
print(
    f"  Chance of landing exactly on a stereotype:  {8 / 2 ** len(t.ITEMS):.1%}  (8 stereotypes among {2 ** len(t.ITEMS)} possible answer patterns)"
)

### 🔍 Look inside: the scoring

Both scores count differing answers (`hamming`). **Accuracy** compares with the original person. **Distance to the nearest stereotype** compares with all 8 stereotypes and keeps the smallest count. The chance level tries all 512 possible answer patterns and averages their distance.

In [ ]:
show_code(t.hamming)
show_code(t.accuracy)
show_code(t.distance_to_stereotype)
show_code(t.chance_stereotype_distance)

## Part 1: one chain, live

Pick one person, the models and the temperature. With 2+ models they take turns along the chain.

If calls keep failing with *"failed on Google's side (HTTP 500)"*, Google's servers are struggling with that model: switch to `["gemini-3.5-flash-lite"]`.


In [ ]:
DEMO_PROFILE = "random-1"
DEMO_MODELS = ["gemma-4-26b-a4b-it"]
DEMO_TEMPERATURE = 1.0
DEMO_CHAIN_LENGTH = 8

original = PROFILES[DEMO_PROFILE]
print("START:", t.profile_to_text(original), "\n")
for r in t.run_chain(caller, original, DEMO_MODELS, DEMO_CHAIN_LENGTH, DEMO_TEMPERATURE):
    print(f"--- agent {r['step']} ({r['model']}, {r['kind']}) ---")
    if r["kind"] == "survey" and r["answers"]:
        marks = []  # ✓ = same as the original person, ✗ = changed
        for i in range(len(original)):
            marks.append("✓" if r["answers"][i] == original[i] else "✗")
        marks = " ".join(marks)
        print(f"answers {''.join(map(str, r['answers']))}  vs original {''.join(map(str, original))}")
        print(
            f"        {marks}  accuracy {r['accuracy']:.0%}, {r['stereotype_distance']} away from nearest stereotype\n"
        )
    elif r["kind"] == "survey":
        print("could not read the answers:", r["raw"][:200], "\n")
    else:
        print(r["raw"].strip(), "\n")

### 🔍 Look inside: the chain

The cell above calls `t.run_chain`. For every agent it:
1. picks the model (with 2 models they take turns),
2. odd agents: sends the **survey prompt** with the latest description, reads the answers and scores them against the **original** person,
3. even agents: sends the **description prompt** with **only** the previous agent's answers; the new description is all the next agent will see,
4. saves a record: step, model asked for, model that answered, time, the full output, and the scores.

Every request goes through `caller.generate`, the only place the agents talk to Google. It sends the prompt with the settings shown in its code (temperature, seed, thinking off for Gemma, a 90-second limit).

In [ ]:
show_code(t.run_chain)
show_code(t.Caller.generate)

## Your turn

Change **one** setting, run 2 chains with your own key, and compare with the big preloaded run.

Ideas: a different person (any name from the table above), `MY_TEMPERATURE = 0.2`, or a different model.

| Model | One 8-agent chain (measured 12 Oct) | Free calls per day |
|---|---|---|
| `gemma-4-26b-a4b-it` | 22 s | 14,400 |
| `gemini-3.5-flash-lite` | 31 s | 500 |
| `gemini-3-flash-preview` | 109 s | 20 |
| `gemini-3.5-flash` (Google now answers with `gemini-3.6-flash`) | 173 s | 20 |
| `gemini-3.6-flash`, `gemini-3.7-flash` | answered; full chain not timed | 20 |

Speeds change from day to day. Models with 20 calls a day allow 2 chains (8 calls each). Mix two models with e.g. `MY_MODELS = ["gemma-4-26b-a4b-it", "gemini-3.5-flash"]`.

In [ ]:
MY_PROFILE = "random-2"
MY_MODELS = ["gemma-4-26b-a4b-it"]
MY_TEMPERATURE = 0.2
MY_CHAINS = 2
MY_CHAIN_LENGTH = 8

original = PROFILES[MY_PROFILE]
for i in range(MY_CHAINS):
    surveys = [
        r for r in t.run_chain(caller, original, MY_MODELS, MY_CHAIN_LENGTH, MY_TEMPERATURE) if r["kind"] == "survey"
    ]
    last = surveys[-1]
    if last["answers"]:
        print(
            f"chain {i + 1}: {''.join(map(str, original))} -> {''.join(map(str, last['answers']))}  "
            f"accuracy {last['accuracy']:.0%}, {last['stereotype_distance']} away from nearest stereotype"
        )
    else:
        print(f"chain {i + 1}: final answers unreadable")

preloaded = a.survey_table(a.load_chains(Path("results/full_run.jsonl")))
readable = preloaded.dropna(subset=["answers"])
last_step = readable.groupby(["setup", "profile", "temperature", "repeat"])["step"].transform("max")
same = readable[(readable["profile"] == MY_PROFILE) & (readable["step"] == last_step)]
print()
print("big run, same person, final step (all setups):")
print(same.groupby(["setup", "temperature"])[["accuracy", "stereotype_distance"]].mean().round(2))

## Part 2: the full experiment

Each setup answers one of the research questions:

| Setup | Question |
|---|---|
| Gemma 26B all the way | does one model stereotype on its own? |
| Gemini 3.5 Flash-Lite all the way | does a different model family behave differently? |
| Gemma 26B and Gemini 3.1 Flash-Lite taking turns | does mixing models change it? |

> **Model sizes:** a clean "same model, different size" test is not possible on the free tier. Gemini does not publish sizes, and Gemma 4 31B (the bigger sibling of 26B) failed most calls on the free tier when we tested it. With paid access or locally (e.g. Ollama), compare Gemma 4 sizes, or another open family that comes in many sizes.

**This run takes about an hour, so it is preloaded** in `results/full_run.jsonl`. Leave `RUN_FULL_EXPERIMENT = False` in the workshop. Set it to `True` only to run or finish it: finished chains are skipped, so a run stopped by a daily limit can continue the next day.

**What gets saved.** Every chain is one line in the results file (JSON). Each chain saves its settings, a **seed** (one fixed seed per repeat, so a single run can be replayed and repeats still differ) and the library version. Each agent's step saves the **model asked for**, the **model that actually answered** (Google can route a name to a newer model), the **time** (UTC), the full output and the scores. The preloaded run was made before seeds, model versions and times were recorded, so those fields are empty there.


In [ ]:
SETUPS = {
    "Gemma 26B": ["gemma-4-26b-a4b-it"],
    "Gemini 3.5 Flash-Lite": ["gemini-3.5-flash-lite"],
    "Gemma 26B + Gemini 3.1 Flash-Lite": ["gemma-4-26b-a4b-it", "gemini-3.1-flash-lite"],
}
TEMPERATURES = [0.2, 1.0]
CHAIN_LENGTH = 8
REPEATS = 3
RESULTS = Path("results/full_run.jsonl")

needed = t.calls_needed(SETUPS, PROFILES, TEMPERATURES, CHAIN_LENGTH, REPEATS)
pd.DataFrame(
    {
        "calls needed": needed,
        "free daily limit": {m: t.REQUESTS_PER_DAY.get(m, t.DEFAULT_RPD) for m in needed},
    }
).assign(days=lambda d: -(-d["calls needed"] // d["free daily limit"]))

In [ ]:
RUN_FULL_EXPERIMENT = False

if RUN_FULL_EXPERIMENT:
    t.run_experiment(caller, RESULTS, SETUPS, PROFILES, TEMPERATURES, CHAIN_LENGTH, REPEATS)
else:
    print(f"Using preloaded results: {len(RESULTS.read_text().splitlines())} chains in {RESULTS}")

### 🔍 Look inside: the full run

`t.run_experiment` makes the list of every setup × person × temperature × repeat, skips chains already in the results file, runs several chains at the same time, and appends each finished chain to the file as one line. That is why a stopped run can simply be started again.

In [ ]:
show_code(t.run_experiment)

## Part 3: what came out

Step 0 is the original person. Chain length is free to study: step 5 of an 8-agent chain is the same as the end of a 5-agent chain.

In [ ]:
chains = a.load_chains(RESULTS)
df = a.survey_table(chains)
print(len(chains), "chains")
a.plot_accuracy(df, group="setup");

### 🔍 Look inside: from results to a chart

`a.survey_table` turns the chains into a table with one row per survey step (step 0 = the original person). `a.plot_accuracy` averages accuracy over all chains for each group and step, draws one line per group, and adds the dashed chance line. The other charts work the same way.

In [ ]:
show_code(a.survey_table)
show_code(a.plot_accuracy)

In [ ]:
a.plot_accuracy(df, group="temperature");

**The stereotype test.** Distance to the nearest coherent profile. Coherent people start at 0. If random people fall toward 0, the chain is "fixing" them into stereotypes.

In [ ]:
a.plot_stereotype(df, group="profile_type");

In [ ]:
for setup, part in df.groupby("setup"):
    a.plot_stereotype(part, group="profile_type").set_title(f"Stereotype drift: {setup}", loc="left")

##### **Branching trees** (as in the human transmission study). One tree per person: chains that gave the same answers are merged, so a thicker line means more chains took that path. Blue lines are labelled with the traits that flipped (`+quiet` = now *yes*). An orange square means the person landed exactly on a stereotype.

Pick any person from the table above. Add `setup="Gemma 26B"` to see one setup only.

In [ ]:
a.plot_tree(chains, "random-2");

In [ ]:
a.plot_tree(chains, "random-1");

**Coherent people for comparison.** Traits that fit together mostly pass through unchanged: a single straight green line. Only one coherent person changed at all in the big run, and only in 1 of 18 chains.

In [ ]:
a.plot_tree(chains, "anxious organized extravert");

In [ ]:
a.plot_tree(chains, "calm organized introvert");

**Attractors**: the final answer patterns many chains ended up in.

In [ ]:
a.attractors(df)

**Broken outputs**: survey replies we could not read (counted, never guessed), and descriptions that repeat themselves. Low temperature can cause looping on some models.

In [ ]:
a.plot_failures(chains);

**Text drift by meaning.** Each description is compared with the chain's *first written* description using Gemini embeddings, so the switch from the template sentences to natural prose does not count as drift. Preloaded in `results/drift.csv`; delete that file to recompute (one embedding call per chain).

In [ ]:
drift = a.text_drift(caller, chains, cache=Path("results/drift.csv"))
a.plot_text_drift(drift, group="setup");

## Export the data

Two CSV files for R, SPSS or Excel:
- `results/survey_answers.csv`: one row per survey step (step 0 = the original person) with accuracy and distance to the nearest stereotype. The table behind the charts above.
- `results/all_agent_outputs.csv`: one row per agent with everything: setup, person, temperature, repeat, seed, step, survey or description, model asked for, model that answered, time, the full text the agent wrote, the parsed answers and the scores.

In [ ]:
df.to_csv("results/survey_answers.csv", index=False)
everything = a.records_table(chains)
everything.to_csv("results/all_agent_outputs.csv", index=False)
print(f"Saved {len(df)} survey rows and {len(everything)} agent rows.")
everything.head()

## Reuse this for your own study

| To change | Where |
|---|---|
| Questionnaire items and which are reverse-keyed | `ITEMS` and `TRAIT_KEYS` in `telephone.py`. Then update `STEREOTYPES` to match; `python telephone.py` checks it. |
| The people that start a chain | `STEREOTYPES` and `RANDOM_PEOPLE` in `telephone.py`: a name and a list of 0/1 answers, one per item. |
| What the agents are told | `survey_prompt` and `description_prompt` in `telephone.py`. Keep the line asking for JSON in the survey prompt, so answers can be read. |
| Models, mixed chains | `SETUPS` in Part 2. For a new model, add its free limits to `REQUESTS_PER_MINUTE` and `REQUESTS_PER_DAY` in `telephone.py`. |
| Chain length, repeats, temperatures | `CHAIN_LENGTH`, `REPEATS`, `TEMPERATURES` in Part 2. |
| Where results go | `RESULTS` in Part 2. Use a **new file name per study**: a run skips chains already in the file. |

**Before a real study**
1. Run 2 or 3 chains first and read the outputs (Part 1) to check the prompts do what you want.
2. Check the *calls needed* table in Part 2 against the daily limits. A run stopped by a daily limit continues the next day: just run the cell again.
3. On Google's free tier, prompts may be used to improve Google's products and read by human reviewers. Use made-up people only, never real participant data; use a paid account for anything sensitive.
4. Report the exact model names, the model versions that answered, the dates, the temperature and the prompts. All of these are in `all_agent_outputs.csv`.

## Open questions for discussion

- **Memory:** here every agent sees only the previous message. What if agents also saw earlier messages?
- **Dynamic communication:** agents exchanging messages over several rounds, or in a network (star, small world).
- **Comparing with humans:** same profiles, same steps, same scoring as the human transmission study.
- **Reproducibility:** the same pinned model still gives different outputs each run, so report the spread over repeats, not one run.